[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C04_AI_Agents_Course/04_coding_agents/04_coding_agents.ipynb)

# 04 · 编码 Agent 解剖：SWE-bench —— 自建迷你 SWE 任务与评分器

**配套讲解**：`04_讲解.html` ｜ **算力**：CPU 即可 ｜ **依赖**：`pytest`（必须）；`datasets` / `transformers` / `openai`（可选）

SWE-bench [Jimenez 2023] 把"修真实 GitHub issue"变成可执行评测：**issue + repo 快照 → agent 产出 patch → FAIL_TO_PASS（修好了）+ PASS_TO_PASS（没修坏）双重验证**。本 notebook 不下载真实仓库，而是**从零自建一个迷你 SWE 任务**，把这条评测管线的每个零件亲手做一遍：

1. 生成一个带边界 bug 的 tiny repo（3 个文件，tempdir 内）；
2. 实现评分器 `score_patch`：应用补丁（全文件替换 / 极简 unified diff 两种模式）→ subprocess 跑 pytest → 判定 resolved；
3. 用空补丁与 gold 补丁验证评分器本身（评测 harness 先自检，再评模型——模块 03 的纪律）；
4. 接入 LLM 扮演编码 agent（OpenAI API / 本地 Qwen / mock 三级回退），跑 3 次统计 resolved 率；
5. 解剖一条真实 SWE-bench Lite 样本的字段结构；
6. ✏️ 3 道练习：pytest 输出解析、F2P/P2P 判定逻辑、自造一个新任务实例。


In [ ]:
import shutil, subprocess, sys, tempfile
from pathlib import Path

# ---------- 迷你 repo 的三个文件 ----------
# calculator.py: 埋一个真实感的边界 bug —— Python 的 // 是 floor 除法(向负无穷取整),
# 而本模块的契约是"向零取整"(同 C 语言), 负数结果会差 1。
CALCULATOR_BUGGY = '''# 迷你计算器模块(刻意埋了一个边界 bug)。

def add(a, b):
    return a + b


def int_divide(a, b):
    # 契约: 整数除法向零取整(truncate toward zero), 同 C 语言。
    # 例: int_divide(7, 2) == 3; int_divide(-7, 2) == -3
    return a // b
'''

UTILS_OK = '''# 工具函数(本任务中没有 bug)。

def clamp(x, lo, hi):
    return max(lo, min(x, hi))


def mean(xs):
    return sum(xs) / len(xs)
'''

# 验收测试: 前 2 个是 FAIL_TO_PASS(修复前红), 后 3 个是 PASS_TO_PASS(必须保持绿)
TESTS = '''from calculator import add, int_divide
from utils import clamp

# --- FAIL_TO_PASS: 修复前红, 修复后应转绿("修好了") ---

def test_int_divide_negative_dividend():
    assert int_divide(-7, 2) == -3


def test_int_divide_negative_divisor():
    assert int_divide(7, -2) == -3

# --- PASS_TO_PASS: 修复前后都必须是绿的("没修坏") ---

def test_int_divide_positive():
    assert int_divide(7, 2) == 3


def test_add():
    assert add(2, 3) == 5


def test_clamp():
    assert clamp(10, 0, 5) == 5
'''

# issue 文本 = SWE-bench 的 problem_statement: 只描述症状与契约, 不给修法
ISSUE_TEXT = '''# Issue #42: int_divide 对负数操作数返回错误结果

int_divide(-7, 2) 返回 -4, 但文档契约是"向零取整", 应返回 -3。
int_divide(7, -2) 同样返回 -4(期望 -3)。正数输入一切正常。
请修复 calculator.py 中的 int_divide; 不要改变函数签名, 不要影响其他函数。
'''

REPO_DIR = tempfile.mkdtemp(prefix="swe_mini_repo_")
for name, content in [("calculator.py", CALCULATOR_BUGGY),
                      ("utils.py", UTILS_OK),
                      ("test_calculator.py", TESTS)]:
    (Path(REPO_DIR) / name).write_text(content, encoding="utf-8")

print("迷你 repo 已生成:", REPO_DIR)
for p in sorted(Path(REPO_DIR).iterdir()):
    print("  -", p.name)


## 迷你任务 ↔ SWE-bench 实例的字段映射

| 我们刚造的东西 | SWE-bench 实例字段 | 作用 |
|---|---|---|
| `ISSUE_TEXT` | `problem_statement` | agent 唯一的需求来源（真实基准里同样只给 issue 原文） |
| tempdir 里的 3 个文件 | repo @ `base_commit` | agent 的工作现场（真实基准是几十万行的仓库快照） |
| 前 2 个测试 | `FAIL_TO_PASS` | 修复前红 → 修复后绿，证明**修好了** |
| 后 3 个测试 | `PASS_TO_PASS` | 修复前后都绿，证明**没修坏** |
| 下面的 `CALCULATOR_GOLD` | `patch`（gold patch） | 参考答案——但评分**只看测试执行**，不比对 gold |

评分器的流程与官方 harness 一致：**复制干净快照 → 应用补丁 → 分别跑两组测试 → 两组全绿才算 resolved**。两个关键设计：

- **agent 看不到验收测试**（防止对着测试硬编码）——这里由评分器持有测试清单；
- 补丁支持两种模式：**模式 A** 全文件替换（`dict`，对弱模型友好，本课 agent 用它）；**模式 B** 极简 unified diff（`str`，真实基准的格式，我们手写一个最小解析器感受它有多脆）。


In [ ]:
import re

F2P = ["test_calculator.py::test_int_divide_negative_dividend",
       "test_calculator.py::test_int_divide_negative_divisor"]
P2P = ["test_calculator.py::test_int_divide_positive",
       "test_calculator.py::test_add",
       "test_calculator.py::test_clamp"]


def run_tests(repo_dir, test_ids):
    # 逐条跑测试, returncode == 0 视为 pass。每条独立子进程, 互不污染。
    results = {}
    for tid in test_ids:
        proc = subprocess.run(
            [sys.executable, "-m", "pytest", tid, "-q", "--tb=no", "-p", "no:cacheprovider"],
            cwd=repo_dir, capture_output=True, text=True, timeout=60)
        results[tid] = "pass" if proc.returncode == 0 else "fail"
    return results


# ---------- 补丁应用: 模式 A(全文件替换) + 模式 B(极简 unified diff) ----------
_HUNK = re.compile(r"@@ -(\d+)(?:,(\d+))? \+(\d+)(?:,(\d+))? @@")


def apply_unified_diff(repo_dir, diff_text):
    # 极简实现: 只支持规范的 --- / +++ / @@ 头与 +/-/上下文行, 不做模糊匹配。
    # 真实 harness 用 git apply——它对行号偏移、换行符等的容错也同样有限,
    # 这正是弱模型用 diff 格式大量翻车的原因。
    lines = diff_text.splitlines()
    i, n = 0, len(lines)
    while i < n:
        if lines[i].startswith("+++ "):
            path = lines[i][4:].strip()
            path = path[2:] if path.startswith("b/") else path
            target = Path(repo_dir) / path
            src = target.read_text(encoding="utf-8").splitlines()
            out, ptr = [], 0
            i += 1
            while i < n and _HUNK.match(lines[i]):
                start = int(_HUNK.match(lines[i]).group(1)) - 1
                out.extend(src[ptr:start])
                ptr = start
                i += 1
                while i < n and not _HUNK.match(lines[i]) and not lines[i].startswith(("--- ", "+++ ")):
                    body = lines[i]
                    if body.startswith("+"):
                        out.append(body[1:])      # 新增行
                    elif body.startswith("-"):
                        ptr += 1                  # 删除原文一行
                    else:
                        out.append(src[ptr]); ptr += 1   # 上下文行原样保留
                    i += 1
            out.extend(src[ptr:])
            target.write_text("\n".join(out) + "\n", encoding="utf-8")
        else:
            i += 1


def apply_patch(repo_dir, patch):
    if isinstance(patch, dict):        # 模式 A: {相对路径: 新的完整文件内容}
        for rel, content in patch.items():
            (Path(repo_dir) / rel).write_text(content, encoding="utf-8")
    elif isinstance(patch, str):       # 模式 B: unified diff 文本
        apply_unified_diff(repo_dir, patch)
    else:
        raise TypeError("patch 应为 dict(全文件替换) 或 str(unified diff)")


def score_patch(repo_dir, patch, f2p=None, p2p=None):
    # SWE-bench 式评分: 干净副本 → 应用补丁 → F2P 全绿 且 P2P 全绿 ⇒ resolved
    f2p = F2P if f2p is None else f2p
    p2p = P2P if p2p is None else p2p
    work = tempfile.mkdtemp(prefix="swe_mini_eval_")
    shutil.copytree(repo_dir, work, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns("__pycache__", ".pytest_cache"))
    if patch is not None:
        try:
            apply_patch(work, patch)
        except Exception as exc:
            return {"resolved": False, "f2p_passed": "0/%d" % len(f2p),
                    "p2p_passed": "0/%d" % len(p2p), "note": "补丁应用失败: %r" % exc}
    f2p_res = run_tests(work, f2p)
    p2p_res = run_tests(work, p2p)
    nf = sum(v == "pass" for v in f2p_res.values())
    np_ = sum(v == "pass" for v in p2p_res.values())
    return {"resolved": nf == len(f2p) and np_ == len(p2p),
            "f2p_passed": "%d/%d" % (nf, len(f2p)),
            "p2p_passed": "%d/%d" % (np_, len(p2p))}


print("评分器就绪: score_patch(repo_dir, patch) -> {resolved, f2p_passed, p2p_passed}")


In [ ]:
# ---------- 先验证评分器本身(harness 自检), 再让模型上场 ----------

# gold 补丁(参考答案), 两种格式各备一份
CALCULATOR_GOLD = '''# 迷你计算器模块(已修复)。

def add(a, b):
    return a + b


def int_divide(a, b):
    # 契约: 整数除法向零取整(truncate toward zero), 同 C 语言。
    # 例: int_divide(7, 2) == 3; int_divide(-7, 2) == -3
    q = abs(a) // abs(b)
    return q if (a >= 0) == (b >= 0) else -q
'''

GOLD_DIFF = '''--- a/calculator.py
+++ b/calculator.py
@@ -10,1 +10,2 @@
-    return a // b
+    q = abs(a) // abs(b)
+    return q if (a >= 0) == (b >= 0) else -q
'''

res_empty = score_patch(REPO_DIR, None)
print("空补丁        :", res_empty)
assert res_empty["resolved"] is False, "空补丁不应 resolved"
assert res_empty["f2p_passed"] == "0/2" and res_empty["p2p_passed"] == "3/3", \
    "空补丁应该是 F2P 全红、P2P 全绿——否则任务/环境本身就坏了"

res_full = score_patch(REPO_DIR, {"calculator.py": CALCULATOR_GOLD})
print("gold(全文件)  :", res_full)
assert res_full["resolved"] is True, "gold 补丁(模式 A)必须 resolved"

res_diff = score_patch(REPO_DIR, GOLD_DIFF)
print("gold(diff)    :", res_diff)
assert res_diff["resolved"] is True, "gold 补丁(模式 B)必须 resolved"

print("\n评分器自检通过 ✓  (空补丁→未解决; gold→resolved; 两种补丁模式一致)")


## Agent 环节：LLM 读 issue → 产出补丁 → 评分

我们实现的是最小的 **Agentless 式管线** [Xia 2024]：定位（本任务免费送——单文件）→ 修复（LLM 重写整个文件，即模式 A）→ 验证（评分器）。没有任何自主循环——对小模型来说，这恰恰是最稳的设计（讲解第 4 节："更多 agency 不一定更高分"）。

`call_llm` 按优先级三级回退：

| 后端 | 触发条件 | 资源 |
|---|---|---|
| OpenAI API（`gpt-4o-mini`） | 设置了 `OPENAI_API_KEY` | 每次调用约 1k token |
| 本地 `Qwen/Qwen2.5-1.5B-Instruct` | 手动把 `USE_LOCAL_LLM = True` | 首次下载约 3.1 GB；CPU 上每次生成约 1–3 分钟 |
| **mock 回退** | 默认（完全离线） | 返回一份脚本化的、会通过的修复回复——只用于演示管线打通 |

⚠️ **教学点**：mock 的 3/3 没有任何信息量（答案是写死的）。换成真模型后，**1.5B 小模型很可能 0/3**——典型死法：没按要求输出完整文件、顺手改坏 `add`（被 P2P 抓住）、或根本不理解 floor 与 truncate 的区别。"能跑通管线"和"能修对 bug"之间隔着一道**能力门槛**，SWE-bench 上 2023 年 1.96% → 2025 年 70%+ 的跨越就是这道门槛被碾过的过程。


In [ ]:
import os

USE_LOCAL_LLM = False   # 改为 True 使用本地 Qwen2.5-1.5B-Instruct(下载约 3.1GB, CPU 可跑但慢)
_LOCAL_PIPE = None

# mock 回退: 一份脚本化的"模型回复", 含一个会通过的补丁(仅用于离线演示管线)
MOCK_RESPONSE = '''诊断: Python 的 `//` 是 floor 除法(向负无穷取整), 负数结果比"向零取整"小 1。
修复: 先用绝对值做整除, 再按操作数符号还原。

```python
# 迷你计算器模块(已修复)。

def add(a, b):
    return a + b


def int_divide(a, b):
    # 契约: 整数除法向零取整(truncate toward zero), 同 C 语言。
    sign = 1 if (a >= 0) == (b >= 0) else -1
    return sign * (abs(a) // abs(b))
```
'''


def call_llm(prompt, seed=0):
    # 优先级: OPENAI_API_KEY > 本地 Qwen > mock(离线脚本化回复)
    if os.environ.get("OPENAI_API_KEY"):
        from openai import OpenAI
        client = OpenAI()
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7, seed=seed, max_tokens=600)
        return resp.choices[0].message.content
    if USE_LOCAL_LLM:
        global _LOCAL_PIPE
        if _LOCAL_PIPE is None:
            from transformers import pipeline
            _LOCAL_PIPE = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct")
        msgs = [{"role": "user", "content": prompt}]
        out = _LOCAL_PIPE(msgs, max_new_tokens=600, do_sample=True, temperature=0.7)
        return out[0]["generated_text"][-1]["content"]
    return MOCK_RESPONSE


def extract_last_code_block(text):
    # 取回复中最后一个 ``` 代码块(模型常先复述旧代码再给修复版)
    blocks = re.findall(r"```(?:python)?\s*\n(.*?)```", text, re.S)
    return blocks[-1] if blocks else None


AGENT_PROMPT = '''你是一个软件工程 agent。阅读下面的 issue 与源文件, 修复其中的 bug。
要求: 只输出一个 ```python 代码块, 块内是修复后的 **完整** calculator.py 文件, 不要省略任何函数。

## Issue
{issue}

## calculator.py 当前内容
```python
{code}
```
'''

prompt = AGENT_PROMPT.format(issue=ISSUE_TEXT, code=CALCULATOR_BUGGY)

trials = []
for t in range(3):
    resp = call_llm(prompt, seed=t)
    code_new = extract_last_code_block(resp)
    if code_new is None:
        trials.append({"resolved": False, "note": "未输出代码块(格式遵循失败)"})
    else:
        trials.append(score_patch(REPO_DIR, {"calculator.py": code_new}))
    print(f"trial {t}: {trials[-1]}")

n_resolved = sum(bool(r.get("resolved")) for r in trials)
print(f"\nresolved 率: {n_resolved}/3")
if not os.environ.get("OPENAI_API_KEY") and not USE_LOCAL_LLM:
    print("(当前是 mock 回退: 3/3 是写死的, 无信息量——换真模型才是实验)")


### 怎么读这个 3 次试验的结果

- **mock = 3/3**：管线通了，但没有任何能力信息（答案写死）。
- **真模型 k/3**：这是一个 n=3 的伯努利估计，噪声极大——SWE-bench 官方报告 pass@1 也只跑一次，所以榜单上 2 分内的差距基本不可分辨（讲解第 3 节的标准误公式）。要量**可靠性**而非碰运气，需要 `pass^k`（k 次全成）这类指标——模块 06 展开。
- **小模型 0/3 本身是数据点**：失败模式（没输出代码块 / P2P 被改坏 / F2P 仍红）的分布，比一个总分更能告诉你瓶颈在哪。这就是讲解第 5 节"把端到端分数拆开归因"的微缩版。

## 真实 SWE-bench 样本解剖

迷你任务的每个零件都对应真实基准的一个字段。下面加载 `princeton-nlp/SWE-bench_Lite`（300 条精简子集）的第 1 条看看真实形态——没装 `datasets` 或离线时，回退到内嵌的演示样例（`django__django-11099`，字段有截断）。


In [ ]:
# 内嵌演示样例: SWE-bench Lite 中的真实实例 django__django-11099(字段有截断/简化)
FALLBACK_SAMPLE = {
    "instance_id": "django__django-11099",
    "repo": "django/django",
    "base_commit": "d26b2424437dabeeca94d7900b37d2df4410da0c",
    "problem_statement": ("UsernameValidator allows trailing newline in usernames. "
        "ASCIIUsernameValidator and UnicodeUsernameValidator use the regex r'^[\\w.@+-]+$'. "
        "The intent is to only allow alphanumeric characters as well as . @ + -. Because of "
        "Python regex behavior of '$', it will also accept usernames ending with a trailing "
        "newline. The fix is to use r'\\A[\\w.@+-]+\\Z' instead. [内嵌样例, 有截断]"),
    "patch": ("--- a/django/contrib/auth/validators.py\n"
              "+++ b/django/contrib/auth/validators.py\n"
              "@@ ... @@\n"
              "-    regex = r'^[\\w.@+-]+$'\n"
              "+    regex = r'\\A[\\w.@+-]+\\Z'\n"
              "[两个 Validator 类各改一处, 有截断]"),
    "test_patch": ("--- a/tests/auth_tests/test_validators.py\n"
                   "+++ b/tests/auth_tests/test_validators.py\n"
                   "[新增'用户名带尾随换行应被拒绝'的断言, 有截断]"),
    "FAIL_TO_PASS": ('["test_ascii_validator (auth_tests.test_validators.UsernameValidatorsTests)", '
                     '"test_unicode_validator (auth_tests.test_validators.UsernameValidatorsTests)"]'),
    "PASS_TO_PASS": '["test_help_text (auth_tests.test_validators.UsernameValidatorsTests)", "...截断"]',
}

sample, src_note = None, ""
try:
    from datasets import load_dataset
    ds = load_dataset("princeton-nlp/SWE-bench_Lite", split="test")
    sample = ds[0]
    src_note = "来自 HuggingFace, 共 %d 条" % len(ds)
except Exception as exc:
    sample = FALLBACK_SAMPLE
    src_note = "加载失败(%s), 使用内嵌演示样例" % type(exc).__name__

print("样本来源:", src_note, "\n")
for key in ["instance_id", "repo", "base_commit", "problem_statement",
            "patch", "test_patch", "FAIL_TO_PASS", "PASS_TO_PASS"]:
    val = str(sample.get(key, "(缺字段)"))
    shown = val[:300].rstrip()
    print("=== %s ===" % key)
    print(shown + (" …[截断, 共 %d 字符]" % len(val) if len(val) > 300 else ""))
    print()

# 读法提示:
# - problem_statement 常常不写复现步骤(欠定问题 → Verified 要人工筛的原因之一);
# - patch 是 gold 参考, 评分不比对它, 只看测试执行;
# - FAIL_TO_PASS/PASS_TO_PASS 是字符串化的测试 id 列表——和我们迷你评分器的 F2P/P2P 一模一样。


## ✏️ 练习 1：解析 pytest 末行摘要

评分器目前靠 returncode 判 pass/fail，但真实 harness 经常要从**文本输出**里提取计数（比如一次跑全部测试时）。pytest 的末行摘要形如：

```
==== 2 failed, 3 passed in 0.12s ====
==== 5 passed in 0.03s ====
==== 1 failed, 2 errors in 0.05s ====
```

实现 `parse_pytest_output(text)`：返回三元组 `(passed, failed, errors)`，没出现的计数记 0。

**提示**：从最后一行往前找含 `passed`/`failed`/`error` 的行，用 `re.findall(r"(\d+) (passed|failed|error)", line)`——注意它对 `errors` 也能匹配到 `error` 前缀。10 行以内可完成。


In [ ]:
def parse_pytest_output(text):
    # TODO: 从 pytest 输出的末行摘要提取 (passed, failed, errors)
    # 步骤: ① 按行倒序找第一个含 passed/failed/error 的行
    #       ② re.findall(r"(\d+) (passed|failed|error)", line) 填计数
    #       ③ 返回 (passed, failed, errors), 缺省为 0
    raise NotImplementedError


In [ ]:
# --- 练习 1 自测 ---
s1 = "collected 5 items\n\ntest_calculator.py ..FF.\n\n==== 2 failed, 3 passed in 0.12s ===="
s2 = "==== 5 passed in 0.03s ===="
s3 = "==== 1 failed, 2 errors in 0.05s ===="

assert parse_pytest_output(s1) == (3, 2, 0), "样式1: failed+passed 混合"
assert parse_pytest_output(s2) == (5, 0, 0), "样式2: 只有 passed"
assert parse_pytest_output(s3) == (0, 1, 2), "样式3: errors(复数)也要数到"
print("✅ 练习 1 通过")


## ✏️ 练习 2：F2P/P2P 判定逻辑

把"双重验证"的判定从评分器里抽出来单独实现。给定**修复前**与**修复后**两份测试结果字典（`test_id -> "pass"/"fail"`），实现 `f2p_p2p_verdict(before, after)`，返回是否 resolved：

- 修复前 `fail` 的测试就是该实例的 **F2P 集**：必须**全部转绿**；
- 修复前 `pass` 的测试就是 **P2P 集**：必须**没有任何退化**；
- `after` 中缺失的测试按 fail 处理（测试被删 = 作弊嫌疑）。

**提示**：先从 `before` 划分两个集合，再各自 `all(...)`，5–8 行可完成。


In [ ]:
def f2p_p2p_verdict(before, after):
    # TODO: 实现 resolved 判定
    # f2p = before 中 fail 的测试; p2p = before 中 pass 的测试
    # 返回: after 中 f2p 全部 pass 且 p2p 全部 pass(缺失视为 fail)
    raise NotImplementedError


In [ ]:
# --- 练习 2 自测 ---
before = {"t_f2p_a": "fail", "t_f2p_b": "fail", "t_p2p_a": "pass", "t_p2p_b": "pass"}

case_fixed  = {"t_f2p_a": "pass", "t_f2p_b": "pass", "t_p2p_a": "pass", "t_p2p_b": "pass"}
case_notfix = {"t_f2p_a": "pass", "t_f2p_b": "fail", "t_p2p_a": "pass", "t_p2p_b": "pass"}
case_broke  = {"t_f2p_a": "pass", "t_f2p_b": "pass", "t_p2p_a": "pass", "t_p2p_b": "fail"}
case_delete = {"t_f2p_a": "pass", "t_f2p_b": "pass", "t_p2p_a": "pass"}  # 删测试

assert f2p_p2p_verdict(before, case_fixed) is True,  "修好且没修坏 → resolved"
assert f2p_p2p_verdict(before, case_notfix) is False, "有 F2P 仍红 → 没修好"
assert f2p_p2p_verdict(before, case_broke) is False,  "P2P 退化 → 修坏了"
assert f2p_p2p_verdict(before, case_delete) is False, "测试消失按 fail 处理"
print("✅ 练习 2 通过")


## ✏️ 练习 3：自造一个新任务实例

任务构造是 SWE-bench 最核心的工程（讲解第 2 节）。现在你来当**基准作者**：迷你 repo 的 `utils.mean` 有一个潜伏缺陷——`mean([])` 抛 `ZeroDivisionError`，而契约要求空列表返回 `0.0`。把它做成一个新的任务实例：

1. `TEST_UTILS_V2`：新测试文件（写入 repo 的 `test_utils.py`），至少 1 个 F2P 测试（`mean([]) == 0.0`）+ 至少 1 个 P2P 测试（如 `mean([1,2,3]) == 2.0`）；
2. `UTILS_GOLD_V2`：gold 补丁 = 修复后的**完整** `utils.py`（`clamp` 原样保留）；
3. 实现 `build_task2()`：复制 `REPO_DIR` 到新 tempdir、写入新测试文件、返回新目录路径；
4. 按你的测试函数命名核对 `F2P_2` / `P2P_2`。

自测会执行 SWE-bench 构造管线的**执行过滤**：空补丁必须不 resolved（F2P 真的红）、gold 补丁必须 resolved（任务真的可解）——这正是原版 SWE-bench 做了、但没做够的那道质检（讲解第 3 节）。


In [ ]:
TASK2_ISSUE = '''# Issue #43: mean([]) 抛 ZeroDivisionError

utils.mean([]) 会崩溃。契约: 空列表应返回 0.0。
请修复 utils.py 中的 mean, 不要改动 clamp。
'''

# TODO 1/3: 新测试文件(将写入 repo 的 test_utils.py)
TEST_UTILS_V2 = '''from utils import mean

# TODO: 写测试函数: test_mean_empty(F2P), test_mean_basic(P2P)
'''

# TODO 2/3: gold 补丁 = 修复后的完整 utils.py(clamp 原样保留)
UTILS_GOLD_V2 = '''
'''

# TODO 3/3: 如改了测试函数名, 同步这两个列表
F2P_2 = ["test_utils.py::test_mean_empty"]
P2P_2 = ["test_utils.py::test_mean_basic", "test_calculator.py::test_clamp"]


def build_task2():
    # TODO: 复制 REPO_DIR 到新 tempdir(忽略 __pycache__),
    #       把 TEST_UTILS_V2 写为 test_utils.py, 返回新目录路径
    raise NotImplementedError


In [ ]:
# --- 练习 3 自测(= 任务构造的"执行过滤") ---
task2_dir = build_task2()

r_empty = score_patch(task2_dir, None, f2p=F2P_2, p2p=P2P_2)
print("空补丁 :", r_empty)
assert r_empty["resolved"] is False, "空补丁不应 resolved(F2P 必须真的先红)"
assert r_empty["p2p_passed"] == "%d/%d" % (len(P2P_2), len(P2P_2)), "P2P 修复前必须全绿"

r_gold = score_patch(task2_dir, {"utils.py": UTILS_GOLD_V2}, f2p=F2P_2, p2p=P2P_2)
print("gold   :", r_gold)
assert r_gold["resolved"] is True, "gold 补丁必须让任务 resolved(任务必须可解)"
print("✅ 练习 3 通过")


## 📖 参考答案

先自己做，再对照。每题一个 cell。


In [ ]:
# 参考答案 1(先自己做, 再对照)
def parse_pytest_output(text):
    counts = {"passed": 0, "failed": 0, "error": 0}
    for line in reversed(text.strip().splitlines()):
        hits = re.findall(r"(\d+) (passed|failed|error)", line)
        if hits:
            for num, word in hits:
                counts[word] = int(num)
            break
    return (counts["passed"], counts["failed"], counts["error"])


In [ ]:
# 参考答案 2(先自己做, 再对照)
def f2p_p2p_verdict(before, after):
    f2p = [t for t, s in before.items() if s == "fail"]
    p2p = [t for t, s in before.items() if s == "pass"]
    f2p_ok = all(after.get(t) == "pass" for t in f2p)   # 全转绿: 修好了
    p2p_ok = all(after.get(t) == "pass" for t in p2p)   # 不退化: 没修坏(缺失=fail)
    return f2p_ok and p2p_ok


In [ ]:
# 参考答案 3(先自己做, 再对照)
TEST_UTILS_V2 = '''from utils import mean

def test_mean_empty():        # FAIL_TO_PASS: 修复前抛 ZeroDivisionError
    assert mean([]) == 0.0


def test_mean_basic():        # PASS_TO_PASS: 修复前后都应通过
    assert mean([1, 2, 3]) == 2.0
'''

UTILS_GOLD_V2 = '''# 工具函数(已修复 mean 的空列表行为)。

def clamp(x, lo, hi):
    return max(lo, min(x, hi))


def mean(xs):
    if not xs:
        return 0.0
    return sum(xs) / len(xs)
'''

F2P_2 = ["test_utils.py::test_mean_empty"]
P2P_2 = ["test_utils.py::test_mean_basic", "test_calculator.py::test_clamp"]


def build_task2():
    work = tempfile.mkdtemp(prefix="swe_task2_")
    shutil.copytree(REPO_DIR, work, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns("__pycache__", ".pytest_cache"))
    (Path(work) / "test_utils.py").write_text(TEST_UTILS_V2, encoding="utf-8")
    return work


## 小结

你亲手搭完了一条 SWE-bench 式评测管线的全部零件：

- **任务实例** = issue 文本 + 仓库快照 + F2P/P2P 双重验收测试；gold patch 只是参考，评分只看执行；
- **评分器纪律**：干净副本上应用补丁、空补丁必须不 resolved、gold 必须 resolved——harness 先自检，再评模型；
- **补丁格式是接口设计**：全文件替换对弱模型友好，unified diff 精确但脆——这就是 SWE-agent [Yang 2024] ACI 思想的微缩版；
- **能力门槛**：mock 3/3 无信息，小模型 0/3 是真实数据点；失败模式的分布比总分更有诊断价值；
- **任务质量决定测量效度**：练习 3 的"执行过滤"做了，原版 SWE-bench 仍有约 1/3 缺陷任务，才有了 Verified [OpenAI 2024]。

**下一站 模块 05 · Computer Use：截图→动作**——当 agent 的观测从"文本化的代码与测试输出"变成**屏幕像素**、行动从"写补丁"变成**点击与键入**，harness 与评测要怎么跟着变？


---
## 🎯 真实数据胶囊题：真实 MBPP 上的 pass@1：正确解 vs 注入 bug

编码 agent 的指标是 pass@1（解能否通过隐藏测试）。用真实 MBPP，对比官方正确解与一个被注入 bug 的版本的 pass 率，理解为什么必须用**执行**而非看代码来评判。

> 本模块新增的**真实数据**练习：自包含、用真实公开数据把本章方法跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, re
import numpy as np
CACHE=os.path.expanduser("~/.ai_agents_data"); os.makedirs(CACHE,exist_ok=True)
def _f(url,fn,headers=None):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p):
        req=urllib.request.Request(url, headers=headers or {})
        open(p,"wb").write(urllib.request.urlopen(req,timeout=40).read())
    return p
def gsm8k(n=300):
    p=_f("https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl","gsm8k_test.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]
def gold(a): return a.split("####")[-1].strip().replace(",","")
def mbpp(n=100):
    p=_f("https://raw.githubusercontent.com/google-research/google-research/master/mbpp/mbpp.jsonl","mbpp.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]

probs=mbpp(60)
def run_tests(code, tests):
    ns={}
    try:
        exec(code,ns)
        for t in tests: exec(t,ns)
        return True
    except Exception: return False
def inject_bug(code):  # 把 return 改成 return None 之类破坏功能
    return code + "\n# bug injected below\n" + code.split('\n')[1].split('def ')[-1].split('(')[0].strip()
print("准备对比 正确解 vs bug 版")

**练习**：实现 `pass_at_1(probs, mutate)`：对每题用 `mutate(code)` 变换后跑测试，返回通过率。`mutate=lambda c:c`(原样) 应高通过率；破坏性 mutate 应低通过率。

In [ ]:
def pass_at_1(probs, mutate):
    # TODO: 对每题 run_tests(mutate(p['code']), p['test_list'])，返回通过比例
    raise NotImplementedError


In [ ]:
# 自测
clean = pass_at_1(probs[:40], lambda c: c)
broken = pass_at_1(probs[:40], lambda c: "def f(): pass")   # 完全错的解
assert clean > 0.85, "正确解高通过率"
assert broken < 0.1, "破坏解低通过率"
print(f"pass@1: 正确解={clean:.2f}  破坏解={broken:.2f} ✓ (只能靠执行判定)")


### 📖 参考答案

In [ ]:
def pass_at_1(probs, mutate):
    return float(np.mean([run_tests(mutate(p["code"]), p["test_list"]) for p in probs]))
print("✓ pass@1 必须执行得出，看代码'像对的'会被骗")